# Gemma 4 (2B) Fine-Tuning with KerasHub and LoRA (Colab Pro / A100 & H100)

Notebook ported from `main_pretrained_gemma.py` to run on Google Colab Pro (A100 / H100) in full precision (`float32`).

## 1. Install Dependencies and Configure Backend

In [ ]:
!pip uninstall -y -q keras-nlp && pip install -q -U keras-hub keras kagglehub grain

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"

import json
import keras
from keras import layers, ops
import keras_hub
import numpy as np
import tensorflow as tf

# Prevent TensorFlow from pre-allocating GPU VRAM
tf.config.set_visible_devices([], "GPU")

print("Keras version:", keras.__version__)
print("KerasHub version:", keras_hub.__version__)
print("Backend:", keras.backend.backend())

## 2. Kaggle Authentication (for KerasHub Presets)

In [ ]:
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Loaded Kaggle credentials from Colab Secrets.")
except Exception:
    import kagglehub
    kagglehub.login()

## 3. Load `Gemma4CausalLM` (`float32`) and Test Before Fine-Tuning

In [ ]:
gemma_lm = keras_hub.models.Gemma4CausalLM.from_preset(
    "gemma4_2b",
    dtype="float32",
)

gemma_lm.summary()
# Preprocessor: "gemma4_causal_lm_preprocessor"
#  Total params: 5,104,298,467 (19.01 GB)
#  Trainable params: 5,104,186,912 (19.01 GB)
#  Non-trainable params: 111,555 (435.76 KB)

In [ ]:
gemma_lm.compile(sampler="greedy")
print(gemma_lm.generate("A piece of advice", max_length=40))
print(gemma_lm.generate("How can I make brownies?", max_length=40))

## 4. Prepare Fine-Tuning Dataset (`databricks-dolly-15k`)

In [ ]:
# --- START DATA FOR FINE TUNING
PROMPT_TEMPLATE = "[instruction]\n{}[end]\n[response]\n"
RESPONSE_TEMPLATE = "{}[end]"

dataset_path = keras.utils.get_file(
    origin=(
        "https://hf.co/datasets/databricks/databricks-dolly-15k/"
        "resolve/main/databricks-dolly-15k.jsonl"
    ),
)
data = {"prompts": [], "responses": []}
with open(dataset_path) as file:
    for line in file:
        features = json.loads(line)
        if features["context"]:
            continue
        data["prompts"].append(PROMPT_TEMPLATE.format(features["instruction"]))
        data["responses"].append(RESPONSE_TEMPLATE.format(features["response"]))

# --- START OF DATASET
# Shuffle and split as Python dicts so KerasHub uses Grain (pure-Python SentencePiece)
# instead of tf.data + tensorflow_text (which crashes in C++ on Colab Python 3.13).
rng = np.random.default_rng(42)
indices = rng.permutation(len(data["prompts"]))
prompts = [data["prompts"][i] for i in indices]
responses = [data["responses"][i] for i in indices]

val_data = {"prompts": prompts[:200], "responses": responses[:200]}
train_data = {"prompts": prompts[200:], "responses": responses[200:]}

## 5. Enable LoRA, Compile, and Train

In [ ]:
# --- START ENABLE LORA
gemma_lm.backbone.enable_lora(rank=8)
gemma_lm.preprocessor.sequence_length = 512

# --- COMPILE AND TRAIN
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(5e-5),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)
gemma_lm.fit(train_data, validation_data=val_data, batch_size=2, epochs=1)

## 6. Test After Fine-Tuning

In [ ]:
# --- START TEST
print(
    gemma_lm.generate(
        "[instruction]\nHello from the mountain[end]\n"
        "[response]\n",
        max_length=512,
    )
)
print(
    gemma_lm.generate(
        "[instruction]\nWhat is going on?[end]\n"
        "[response]\n",
        max_length=512,
    )
)
print(
    gemma_lm.generate(
        "[instruction]\nWho are you?[end]\n"
        "[response]\n",
        max_length=512,
    )
)